In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()

c:\Users\Lenovo\OneDrive\Desktop\langgraph\myvenv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


True

In [2]:
model=ChatGoogleGenerativeAI( model='gemini-3-flash-preview',
  temperature=1)

In [ ]:
class PromptChainingState(TypedDict):
    title:str
    outline:str
    blog:str

def outline_Generate(state:PromptChainingState)->PromptChainingState:
    title=state['title']

    prompt=f"Generate a detailed outline for a blog post with the title: {title}"
    outline=model.invoke(prompt).content

    state['outline']=outline
    return state

def blog_generator(state:PromptChainingState)->PromptChainingState:
    title=state['title']
    outline=state['outline']

    prompt=f"Generate a blog post based on the following title and outline:\nTitle: {title}\nOutline: {outline}"
    blog=model.invoke(prompt).content

    state['blog']=blog
    return state

In [4]:
graph=StateGraph(PromptChainingState)

graph.add_node('outline_generator',outline_Generate)
graph.add_node('blog_generator',blog_generator)

graph.add_edge(START,'outline_generator')
graph.add_edge('outline_generator','blog_generator')
graph.add_edge('blog_generator',END)

workflow=graph.compile()

In [5]:
initial_state=PromptChainingState(title='The Future of AI in Everyday Life',outline='',blog='')

final_state=workflow.invoke(initial_state)

print(final_state)

{'title': 'The Future of AI in Everyday Life', 'outline': [{'type': 'text', 'text': 'This detailed outline is designed to create a comprehensive, SEO-friendly, and engaging blog post about the integration of Artificial Intelligence into our daily routines.\n\n---\n\n# Title: The Future of AI in Everyday Life\n**Subtitle:** From invisible assistants to personalized medicine: How AI is quietly rewriting the human experience.\n\n---\n\n## I. Introduction\n*   **The Hook:** Start with a "day in the life" comparison (e.g., how we started our mornings 10 years ago vs. today).\n*   **The Current State of AI:** Briefly mention that AI is no longer a "sci-fi" concept; it’s already in our pockets (Siri, Netflix recommendations, Google Maps).\n*   **The Thesis Statement:** AI is transitioning from a tool we *use* to an invisible utility—like electricity—that powers the background of our lives.\n*   **Overview:** A roadmap of the areas the blog will cover (Home, Health, Work, and Ethics).\n\n## II